# Exploratory Experiments

Miscellaneous experiments with word embeddings — analogy chains, interpolation, debiasing spot-checks, etc.
Run the setup cells in `FATE.ipynb` first, or use the setup below.

In [ ]:
import plotly.graph_objects as go
import numpy as np
from numpy.linalg import norm
from arsenal import Alphabet
from embedding import Embeddings, normalize_rows, load_vecs, plot_change

with np.load('vecs.npz') as data:
    emb = Embeddings(normalize_rows(data['vec']), Alphabet(data['voc']))
analogy = emb.analogy

gender_pairs = """
she he
woman man
herself himself
her him
hers his
gal guy
girl boy
girls boys
female male
females males
girls guys
girl guy
"""

deb = emb.debias(gender_pairs, K=10)

diff = norm(emb.vec - deb.vec, axis=1)
top = np.argsort(-diff)

In [ ]:
emb.plot_analogy('man', 'woman', 'john', n=10)

In [ ]:
#analogy("man :: stud -> woman", n=3)
#analogy("john :: dog -> mary")
analogy("tim :: amazing -> hanna")
analogy("portuguese :: vieira -> american")
analogy("tim :: ryan -> tall")

In [ ]:
def _chain(a, b, c, max_length):
    chain = [c]
    for _ in range(max_length):
        [d] = emb._analogy(a, b, c, n=1)
        if d in chain: break
        chain.append(d)
        c = d
    return chain[1:]

def chain(a, b, c, max_length=20):
    cs = _chain(a, b, c, max_length)
    print(a, b, c, '->'.join(cs))

#print(chain('dog', 'cat', 'loyal'))
#print(chain('man', 'woman', 'athlete'))
chain('man', 'woman', 'genius')
chain('man', 'woman', 'smart')
chain('man', 'woman', 'doctor')
chain('man', 'woman', 'tennis')
chain('man', 'woman', 'golf')
chain('man', 'woman', 'hockey')
#chain('man', 'woman', 'horny')
#chain('man', 'woman', 'handsome')
#chain('woman', 'man', 'handsome')

In [ ]:
def biased_focused(emb, B):
    A = emb.vec @ B.T
    return Embeddings(A, emb.dom)

#deb2 = Debiased.fit(emb, k = 2, G = gender_pairs)
#C = biased_focused(emb, deb2.B)
C = biased_focused(emb, deb.B)

In [ ]:
professions = [
    'caretaker', 'homemaker',
    'doctor', 'nurse', 'programmer', 'teacher', 
    'wife', 'husband', 'soldier', 'salesperson', 'analyst', 'therapist',
    'trainer', 'instructor', 'ceo', 'assistant', 'telemarketer', 'bartender', 'clerk',
    'designer', 'father', 'mother', 'scientist', 'manager', 'boss', 'self-employed',
]

gen = [
    'boy', 'girl', 'brother', 'sister', 'mom', 'dad', 'daughter', 'son', 'mother', 'father',
]

misc = [ 
    'man', 'woman', 
]

misc += professions
misc += gen

for w in misc:
    print(w, ':', emb.most_similar(emb(w), n=2)[1], '->', C.most_similar(C(w), n=2)[1])

emb._plot_paths([misc])
C._plot_paths([misc]);

In [ ]:
fig = go.Figure(go.Scatter(y=diff[top], mode='lines'))
fig.update_layout(width=600, height=300)
fig.show()

In [ ]:
deb.analogy("man :: woman -> dog")
deb.analogy("woman :: man -> dog")
print(deb.most_similar(deb("dog"), n=3)[1:])

In [ ]:
deb.analogy("man :: woman -> cat")
deb.analogy("woman :: man -> cat")
print(deb.most_similar(deb("cat"), n=3)[1:])

In [ ]:
deb.analogy("man :: woman -> him")
deb.analogy("woman :: man -> him")
print(deb.most_similar(deb("him"), n=3)[1:])

In [ ]:
analogy("woman :: man -> menstruating")
analogy("man :: woman -> menstruating")
print('----')
deb.analogy("woman :: man -> menstruating")
deb.analogy("man :: woman -> menstruating")
print(deb.most_similar(deb("menstruating"), n=3)[1:])

In [ ]:
target = 'testosterone'
analogy(f"woman :: man -> {target}")
analogy(f"man :: woman -> {target}")
print('----')
deb.analogy(f"woman :: man -> {target}")
deb.analogy(f"man :: woman -> {target}")
print(deb.most_similar(emb(target), n=3)[1:])
print(deb.most_similar(deb(target), n=3)[1:])

# Can we interpolate between concepts?

## Idea 1

Pick two end points, interpolate their embeddings as by convex combination, find the point which is closest to the interpolated vector.

In [ ]:
def interpolate1(X, Y, n=10, plot=True, force=False):
    x, y = emb(X), emb(Y)
    excl = set()
    path = [X]
    for a in np.linspace(0, 1, n):
        q = x*(1-a) + a*y
        [w] = emb.most_similar(q, n=1, exclude=excl)
        if w not in path: path.append(w)
        if force: excl.add(w)            
    if plot: emb.plot_paths([path])
    print(' -> '.join(path))

In [ ]:
interpolate1("boss", "customer", plot=False)
interpolate1("boss", "customer", plot=False, force=True)

interpolate1("boss", "employee", plot=False)

Why isn't the plot one-dimensional?  Is it because taking one-best is nonlinear?

## Idea 2

Bisection: pick a midpoint.

In [ ]:
def dedup(xs):
    s = set()
    for x in xs:
        if x not in s: yield x
        s.add(x)

def bisect(X, Y, n=10, plot=True, force=False):
    path = [X] + list(_bisect(X, Y)) + [Y]
    print(' -> '.join(dedup(path)))

def _bisect(X, Y):
    x = emb(X); y = emb(Y)
    [W] = emb.most_similar((x + y) / 2, n=1)
    if X != W and W != Y:
        yield from _bisect(X, W)
        yield W
        yield from _bisect(W, Y)

In [ ]:
bisect("boss", "customer", plot=False)
bisect("boss", "employee", plot=False)

This didn't work at all!